# 8. SP whole-data validation (Henderson-style)

Fits the **same SP model used in the session-wise drift analysis** (notebook 2, "ori" model: 16 pRF-pooled orientation x spatial-frequency energies + low-pass + high-pass residual energy + constant, OLS per voxel) **once on all available data**, and evaluates it the way Henderson et al. (2023) evaluate the texture model:

1. betas / 300, z-scored per voxel within each session
2. averaged over repetitions of each image
3. trained on all non-shared images, validated on the shared-1000 images
4. validation R2 per voxel

ROIs: NSD `prf-visualrois` (V1 = V1v+V1d, V2 = V2v+V2d, V3 = V3v+V3d, hV4).

Output: `V1_Drift/repDrift_expand/wholedata/SP_wholedata_sub{1..8}.npy` (dict per ROI: `voxel_index` = whole-brain flat index, `val_r2`, `n_train`, `n_val`, `n_sessions`).

The figure comparing SP with the TS model is in `TS_Model/notebooks/fig3_TS_SP_comparison.ipynb`.

Runtime: ~30-60 min per subject (one ~2 GB full-brain betas file per session); no GPU needed, High-RAM recommended. Re-running skips finished subjects.

In [ ]:
# ==== SP pipeline configuration -- RUN THIS CELL FIRST ====
# Must match ACTIVE_CONFIG in 1_nsd_prf_sampling_from_pyramid.ipynb.
# Every path in this notebook is built from PRF_ROOT (pRF-sampled features)
# and RES_ROOT (regression / drift results), so the two image configurations
# never read or overwrite each other's files.
SP_CONFIGS = {
    "256_no_background": dict(
        prf_folder="/content/drive/MyDrive/V1_Drift/prfsample_expand",
        save_dir="/content/drive/MyDrive/V1_Drift/repDrift_expand",
        background=False,
    ),
    "256_with_background": dict(
        prf_folder="/content/drive/MyDrive/V1_Drift/prfsample_expand_with_background",
        save_dir="/content/drive/MyDrive/V1_Drift/repDrift_expand_with_background",
        background=True,
    ),
}

# >>> Choose which configuration to run <<<
ACTIVE_CONFIG = "256_with_background"

PRF_ROOT = SP_CONFIGS[ACTIVE_CONFIG]["prf_folder"]
RES_ROOT = SP_CONFIGS[ACTIVE_CONFIG]["save_dir"]
print(f"Configuration: {ACTIVE_CONFIG}\n  pRF samples: {PRF_ROOT}\n  results:     {RES_ROOT}")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ==== SP (steerable pyramid 4 ori x 4 SF) whole-data fit, Henderson-style validation ====
# Same model as the session-wise SP regression (2_nsd_regresion_prf_split.ipynb, the "ori" model):
#   X = 16 pRF-pooled orientation x SF energies + low-pass + high-pass + constant, OLS per voxel.
# Same evaluation as Henderson et al. (2023):
#   betas /300, z-scored per voxel within each session, averaged over repetitions of each image;
#   train on all non-shared images, validate on the shared-1000 images; R2 on the validation set.
import os, gc, time, numpy as np, scipy.io as sio, h5py, nibabel as nib

NSD_BASE  = "/content/drive/MyDrive/V1_Drift/NSD"
PRF_DIR   = PRF_ROOT
SAVE_DIR  = RES_ROOT + "/wholedata"
os.makedirs(SAVE_DIR, exist_ok=True)

N_SESS_SUBJ = [40, 40, 32, 30, 40, 32, 40, 30]
TRIALS_PER_SESS = 750
REGION_ROIS = {1: [0, 1], 2: [2, 3], 3: [4, 5], 4: [6]}          # V1, V2, V3, hV4 (prf-visualrois labels 1..7)
REGION_NAMES = {1: "V1", 2: "V2", 3: "V3", 4: "hV4"}

def load_session_betas(isub, isess):
    fn = os.path.join(NSD_BASE, f"subj{isub:02d}", "betas", f"betas_session{isess:02d}.nii.gz")
    b = nib.load(fn).get_fdata(dtype=np.float32) / 300.0
    return b.reshape(-1, b.shape[-1])                                 # (n_vox_total, 750), C-order like the SP code

def r2_score(y, yhat):
    ss_res = np.sum((y - yhat) ** 2); ss_tot = np.sum((y - y.mean()) ** 2)
    return 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan

def sp_wholedata_fit(isub, max_sessions=None):
    t0 = time.time()
    # ---- design: which 73k image was shown on each trial, and which are shared ----
    des = sio.loadmat(os.path.join(NSD_BASE, "nsddata", "experiments", "nsd", "nsd_expdesign.mat"))
    masterordering = des["masterordering"].ravel() - 1
    trial_img = des["subjectim"][isub - 1, masterordering]          # 73k image ID per trial (1-based)
    shared = set(des["sharedix"].ravel().tolist())                   # shared-1000 image IDs (1-based)

    # ---- sessions that exist on Drive ----
    n_sess = N_SESS_SUBJ[isub - 1] if max_sessions is None else min(max_sessions, N_SESS_SUBJ[isub - 1])
    sessions = [s for s in range(1, n_sess + 1) if os.path.exists(
        os.path.join(NSD_BASE, f"subj{isub:02d}", "betas", f"betas_session{s:02d}.nii.gz"))]
    print(f"S{isub}: using {len(sessions)} sessions ({sessions[0]}..{sessions[-1]})")

    # ---- voxels of all four regions ----
    labels = nib.load(os.path.join(NSD_BASE, f"subj{isub:02d}", "func1pt8mm", "roi", "prf-visualrois.nii.gz")).get_fdata().ravel()
    roi_vox = {r: np.flatnonzero(labels == (r + 1)) for r in range(7)}          # same order as roiInd in the SP code
    all_vox = np.concatenate([roi_vox[r] for r in range(7)])

    # ---- load, z-score within session, collect ----
    Y, img_of_trial = [], []
    for s in sessions:
        b = load_session_betas(isub, s)[all_vox, :]
        mu, sd = b.mean(axis=1, keepdims=True), b.std(axis=1, keepdims=True)
        Y.append(((b - mu) / np.where(sd > 0, sd, 1)).astype(np.float32))
        img_of_trial.append(trial_img[(s - 1) * TRIALS_PER_SESS: s * TRIALS_PER_SESS])
        del b; gc.collect()
        print(f"  session {s:02d} loaded ({time.time()-t0:.0f}s)")
    Y = np.concatenate(Y, axis=1); img_of_trial = np.concatenate(img_of_trial)

    # ---- average over repetitions of each image ----
    uniq, inv = np.unique(img_of_trial, return_inverse=True)
    order = np.argsort(inv, kind="stable")
    counts = np.bincount(inv)
    starts = np.concatenate([[0], np.cumsum(counts)[:-1]])
    Yavg = (np.add.reduceat(Y[:, order], starts, axis=1) / counts[None, :]).astype(np.float32)
    is_val = np.array([u in shared for u in uniq])
    print(f"  unique images: {len(uniq)} (train {np.sum(~is_val)}, val {np.sum(is_val)})")
    del Y; gc.collect()

    out = {}
    vox_offset = {r: (sum(len(roi_vox[q]) for q in range(r)), len(roi_vox[r])) for r in range(7)}
    for region, rois in REGION_ROIS.items():
        with h5py.File(os.path.join(PRF_DIR, f"prfSampleStim_v{region}_sub{isub}.h5"), "r") as f:
            all_imgs = f["allImgs"][:].astype(int)
            row_of = {im: i for i, im in enumerate(all_imgs)}
            rows = np.array([row_of.get(u, -1) for u in uniq]); have = rows >= 0
            vidx, r2s = [], []
            for roinum in rois:
                lev    = f[f"prfSampleLev/roi_{roinum}"][:]            # (n_img, n_vox, n_levels+2)
                levori = f[f"prfSampleLevOri/roi_{roinum}"][:]         # (n_img, n_vox, n_levels, n_ori)
                n_img, nv, nl, no = levori.shape
                o, n = vox_offset[roinum]
                assert n == nv, f"S{isub} roi {roinum}: {n} voxels in betas vs {nv} in pRF file"
                tr, va = have & ~is_val, have & is_val
                for v in range(nv):
                    Xo = levori[rows, v].reshape(len(rows), nl * no, order="F")       # same column order as SP code
                    X = np.concatenate([Xo, lev[rows, v, -2:], np.ones((len(rows), 1), np.float32)], axis=1)
                    y = Yavg[o + v]
                    coef, *_ = np.linalg.lstsq(X[tr].astype(np.float64), y[tr].astype(np.float64), rcond=None)
                    r2s.append(r2_score(y[va], X[va] @ coef))
                vidx.append(roi_vox[roinum])
                del lev, levori; gc.collect()
        out[REGION_NAMES[region]] = dict(voxel_index=np.concatenate(vidx), val_r2=np.array(r2s, np.float32),
                                         n_train=int(np.sum(have & ~is_val)), n_val=int(np.sum(have & is_val)),
                                         n_sessions=len(sessions))
        r = out[REGION_NAMES[region]]["val_r2"]
        print(f"  {REGION_NAMES[region]}: {len(r)} voxels, median R2 {np.nanmedian(r):.4f}")

    fn = os.path.join(SAVE_DIR, f"SP_wholedata_sub{isub}.npy")
    np.save(fn, out, allow_pickle=True)
    print(f"S{isub} saved -> {fn}  ({(time.time()-t0)/60:.1f} min)")
    return out

# ---- run (skips subjects already done, so it can be re-run after a disconnect) ----
for isub in range(1, 9):
    if os.path.exists(os.path.join(SAVE_DIR, f"SP_wholedata_sub{isub}.npy")):
        print(f"S{isub}: already done"); continue
    sp_wholedata_fit(isub)
